# Online Retail — Câu 8: Xóa mặt hàng POSTAGE

**Võ Thiện Linh — MSSV: 6451071041**

Tạo one-hot basket sau khi làm sạch dữ liệu và xóa cột `POSTAGE` trước khi khai thác tập phổ biến.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
uk = df.loc[df['Country'].eq('United Kingdom')]
grouped_quantity = uk.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_index = pd.Index(grouped_quantity['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
row_position = {invoice: i for i, invoice in enumerate(invoice_index)}
sparse_columns = {}
for description, rows in grouped_quantity.groupby('Description', sort=False):
    values = np.zeros(len(invoice_index), dtype=bool)
    positions = rows['InvoiceNo'].map(row_position).to_numpy(dtype=np.intp)
    values[positions] = rows['Quantity'].to_numpy() > 0
    sparse_columns[description] = pd.arrays.SparseArray(values, fill_value=False)
basket_onehot = pd.DataFrame(sparse_columns, index=invoice_index)
if 'POSTAGE' in basket_onehot.columns:
    postage_invoices = int(basket_onehot['POSTAGE'].sum())
    basket_onehot = basket_onehot.drop(columns='POSTAGE')
else:
    postage_invoices = 0
print(f'Số hóa đơn có POSTAGE trước khi xóa cột: {postage_invoices:,}')
print('POSTAGE còn trong ma trận:', 'POSTAGE' in basket_onehot.columns)
print(f'One-hot basket sau khi bỏ POSTAGE: {basket_onehot.shape[0]:,} hóa đơn × {basket_onehot.shape[1]:,} mặt hàng')
print('Một số cột còn lại:')
display(basket_onehot.iloc[:6, :8].sparse.to_dense().astype(int))
assert 'POSTAGE' not in basket_onehot.columns